# Data Access Audit Analysis

Review data-access telemetry for compliance investigations and insider-risk triage.

In [ ]:
from datetime import datetime, timedelta
import clickhouse_connect
import matplotlib.pyplot as plt
import os
from clario360 import Client

sdk = Client.from_env()
DAYS_BACK = int(os.environ.get('CLARIO360_AUDIT_WINDOW_DAYS', '14'))
client = clickhouse_connect.get_client(
    host=os.environ['CLICKHOUSE_HOST'],
    port=int(os.environ.get('CLICKHOUSE_PORT', '9000')),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'clario360'),
    username=open('/etc/clario360/credentials/clickhouse_user').read().strip(),
    password=open('/etc/clario360/credentials/clickhouse_password').read().strip(),
)


## Fetch access events

This query is suitable for routine audit reviews and can be narrowed by user or source if needed.

In [ ]:
query = '''
SELECT user_email, action, resource_type, count() AS event_count
FROM security_events
WHERE tenant_id = %(tenant_id)s
  AND timestamp >= now() - INTERVAL %(days)s DAY
GROUP BY user_email, action, resource_type
ORDER BY event_count DESC
LIMIT 100
'''
sdk.record_data_query('clickhouse', 'Data access audit summary', {'days_back': DAYS_BACK})
df = client.query_df(query, parameters={'tenant_id': sdk.tenant_id, 'days': DAYS_BACK})
df.head(20)

## Visualize concentration of access

High-frequency readers or exporters are useful starting points for audit follow-up.

In [ ]:
pivot = df.head(20).pivot_table(index='user_email', columns='action', values='event_count', fill_value=0)
pivot.plot(kind='bar', stacked=True, figsize=(12, 5))
plt.title('Top Data Access Actors')
plt.xlabel('User Email')
plt.ylabel('Event Count')
plt.tight_layout()
plt.show()
df.groupby('resource_type')['event_count'].sum().sort_values(ascending=False)
